In [1]:
from typing import TypedDict


class SubGraphState(TypedDict):
    """子图状态"""

    # 主图状态
    main_value: str
    # 子图状态 - 在原本的设计模式中， 是否可以看作是临时变量
    sub_value: str


def subgraph_node_1(state: SubGraphState) -> SubGraphState:
    return {"sub_value": "你好！"}


def subgraph_node_2(state: SubGraphState) -> SubGraphState:
    return {"main_value": f"{state['main_value']} - {state['sub_value']}"}


from langgraph.graph.state import END, START, StateGraph

subgraph = (
    StateGraph(SubGraphState)
    .add_node("subgraph_node_1", subgraph_node_1)
    .add_node("subgraph_node_2", subgraph_node_2)
    .add_edge(START, "subgraph_node_1")
    .add_edge("subgraph_node_1", "subgraph_node_2")
    .add_edge("subgraph_node_2", END)
    .compile()
)


class MainGraphState(TypedDict):
    """主图状态"""

    main_value: str


def main_node(state: MainGraphState) -> MainGraphState:
    return {"main_value": f"{state['main_value']}, main_node"}


main_graph = (
    StateGraph(MainGraphState)
    .add_node("main_node", main_node)
    .add_node("subgraph_node", subgraph)
    .add_edge(START, "main_node")
    .add_edge("main_node", "subgraph_node")
    .add_edge("subgraph_node", END)
    .compile()
)

async for chunk in main_graph.astream({"main_value": "今天天气怎么样?"}, subgraphs=True):
    print(chunk)


((), {'main_node': {'main_value': '今天天气怎么样?, main_node'}})
(('subgraph_node:e52460cd-0cc6-4c37-9003-f0e568a4e616',), {'subgraph_node_1': {'sub_value': '你好！'}})
(('subgraph_node:e52460cd-0cc6-4c37-9003-f0e568a4e616',), {'subgraph_node_2': {'main_value': '今天天气怎么样?, main_node - 你好！'}})
((), {'subgraph_node': {'main_value': '今天天气怎么样?, main_node - 你好！'}})
